# ECG AF Detector
Finds atrial fibrillation in a single-lead ECG, window by window, and reports the AF burden. Code and full results: https://github.com/fady-mostafa-omar/ecg-af-detector

The free CPU runtime is enough for the app.

## 0. Setup (about 1 minute)
Installs the code. The app needs no data download: its example recordings are bundled.

In [1]:
!git clone -q https://github.com/fady-mostafa-omar/ecg-af-detector.git
%cd ecg-af-detector
!pip -q install wfdb==4.3.1 gradio plotly pyyaml awscli

/content/ecg-af-detector
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sphinx 8.2.3 requires docutils<0.22,>=0.20, but you have docutils 0.19 which is incompatible.


## 1. Try the app
Open the `gradio.live` link printed below. The app stays online while this cell runs; press ■ to stop it before running section 2.

In [5]:
import subprocess, time
from google.colab import output
subprocess.run("pkill -f app/app.py", shell=True)
subprocess.Popen("cd /content/ecg-af-detector && python app/app.py > app.log 2>&1", shell=True)
time.sleep(45)
output.serve_kernel_port_as_window(7860)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

## 2. Reproduce the results
Every table and figure in `results/` is written by these scripts. Download the data (CinC 2017: 8,528 recordings; MIT-BIH Arrhythmia Database: about 100 MB; MIT-BIH AF Database: about 0.6 GB) and build the dataset:

In [ ]:
!bash scripts/download_data.sh
!python scripts/build_dataset.py

R peaks, features and 30-s windows for all recordings (about 10 minutes for the R peaks alone on the free CPU runtime), then the gradient-boosting baseline:

In [ ]:
!python scripts/extract_features.py
!python scripts/baseline.py

Optional: retrain the 5-seed CNN ensemble. Use a GPU runtime (about 2 minutes per seed on a T4; hours on a CPU). The trained ensemble is already in `models/afnet_ensemble.pt`; skip this cell to use it.

In [ ]:
# !python scripts/train_cnn.py

In [ ]:
!python scripts/evaluate.py

External test on 45 long Holter recordings (about 245 hours of ECG). This is the slowest step; a GPU runtime helps.

In [ ]:
!python scripts/external_test.py